# Phase 1 — Diagnostic des substitutions ht → gcf

Objectif : produire `substitution_rules_ht2gcf.json`.

**L'étape 2 n'est pas automatisable.** La correction des 300 segments vers le gcf standard (GEREC-2) est un travail humain ; tout ce qui suit en dépend.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.config import CFG


## 1. Extraire 300 segments aléatoires

In [ ]:
from src.phase1_diagnostic import build_parser, cmd_sample, cmd_mine, cmd_audit
cmd_sample(build_parser().parse_args(['sample', '--n', '300']))


## 2. Correction manuelle

Ouvrir `data/diagnostic_to_correct.csv`, remplir la colonne `corrected`, enregistrer sous `data/diagnostic_corrected.csv`.

La colonne `pre_correction` propose une ébauche issue des règles amorces : elle fait gagner du temps mais ne doit jamais être validée sans relecture, plusieurs amorces étant des hypothèses non vérifiées.

## 3-4. Alignement mot-à-mot et fouille des règles

In [ ]:
cmd_mine(build_parser().parse_args(['mine', '--min-support', '5', '--min-confidence', '0.6']))


### Inspection d'un alignement

In [ ]:
from src.rules import align_words, decompose_block
from src.normalize import tokenize

ht  = 'mwen ap manje nan kay la'
gcf = 'an ka manjé adan kaz la'
for b in align_words(tokenize(ht), tokenize(gcf)):
    print(f'{b.op:8s} {b.src} -> {b.tgt}')


## 5. Cas résiduels non couverts

`artifacts/residual_cases.json` liste les divergences écartées. C'est la charge de travail transmise au correcteur neuronal — à lire avant de passer à la Phase 2.

In [ ]:
from src.io_utils import read_json
res = read_json(CFG.residuals_path)
print(res['n_residual_blocks'], 'blocs résiduels')
for r in res['top_unexplained'][:20]: print(r)


In [ ]:
cmd_audit(build_parser().parse_args(['audit', '--show', '20']))
